# 02 · Families, features and the train/test split

Day 2. Starts from `data/processed/cicids2017_clean.parquet` (made on day 1) and:

1. groups labels into families and drops the two that are too small,
2. compares benign and attack traffic on a few features,
3. drops the leaky `Destination Port` and near-duplicate features,
4. ranks the rest with a Random Forest and keeps the top 25,
5. makes a stratified 70/15/15 split and saves it.

Everything here calls `src/ids/features/prepare.py`; the terminal version is
`python -m ids.features.prepare`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from ids.data.clean import FAMILY_COL, LABEL_COL
from ids.features.prepare import (
    LEAKY_FEATURES, RARE_FAMILIES, correlated_features, drop_rare_and_leaky,
    feature_columns, rank_features, save_feature_config, split,
)

TOP_K = 25
SEED = 42
PROCESSED = ROOT / "data/processed"
FIGURES = ROOT / "reports/figures"
FIGURES.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
df = pd.read_parquet(PROCESSED / "cicids2017_clean.parquet")
print(f"{len(df):,} rows x {df.shape[1]} columns")

## 1. Labels and families

Fifteen labels collapse into nine families. Infiltration (36 rows) and Heartbleed (11 rows) are far too small to learn or evaluate, so they are dropped and the README says so.

In [ ]:
pd.crosstab(df[LABEL_COL], df[FAMILY_COL]).replace(0, "")

In [ ]:
fam = df[FAMILY_COL].value_counts()
ax = fam.plot.barh(logx=True, figsize=(8, 4), color="steelblue")
ax.invert_yaxis()
ax.set(xlabel="rows (log scale)", title="Rows per family after cleaning")
for i, v in enumerate(fam):
    ax.text(v, i, f" {v:,}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES / "02_family_counts.png", dpi=120)
print("dropping:", RARE_FAMILIES, "->", int(fam.reindex(RARE_FAMILIES).fillna(0).sum()), "rows")

## 2. Benign vs attack on a few features

Log scale, on a 50k-row sample. Look for features where the families barely overlap: those are the ones the model will lean on.

In [ ]:
data = drop_rare_and_leaky(df)
sample = pd.concat(
    g.sample(min(len(g), 7000), random_state=SEED) for _, g in data.groupby(FAMILY_COL)
)
show = [c for c in ["Flow Duration", "Total Fwd Packets", "Fwd Packet Length Max",
                    "Bwd Packet Length Mean", "Flow IAT Mean", "Init_Win_bytes_forward"]
        if c in sample.columns]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, show):
    sns.boxplot(data=sample, y=FAMILY_COL, x=np.log1p(sample[col].clip(lower=0)),
                ax=ax, fliersize=0, color="lightsteelblue")
    ax.set(xlabel=f"log1p({col})", ylabel="")
plt.tight_layout()
plt.savefig(FIGURES / "02_feature_boxplots.png", dpi=120)

**Negative values.** A few columns have negatives. `Init_Win_bytes_* = -1` means "no window seen" and is meaningful, so we keep it. Tree models don't care about the sign.

In [ ]:
feats = feature_columns(data)
neg = (data[feats] < 0).sum()
neg[neg > 0].sort_values(ascending=False).to_frame("rows < 0")

## 3. Split first, then select

We split **before** choosing features, so the test set has no say in which features are kept. Stratified on family: every split has the same class mix. (A time-based split would be stricter; the README notes this.)

In [ ]:
splits = split(data, seed=SEED)
train = splits["train"]
pd.DataFrame({k: v[FAMILY_COL].value_counts() for k, v in splits.items()})

## 4. Leaky and near-duplicate features

`Destination Port` is already gone (it identifies the attacked service, not the behaviour). Now drop one of each pair of features with |correlation| > 0.98. Keeping both of a twin pair adds nothing and splits importance between them.

In [ ]:
print("leaky, dropped:", LEAKY_FEATURES)
dropped_corr = correlated_features(train, feats)
candidates = [c for c in feats if c not in dropped_corr]
print(f"near-duplicates dropped ({len(dropped_corr)}):", dropped_corr)
print(f"{len(candidates)} candidate features left")

## 5. Random Forest importance

A 100-tree forest on a 200k-row stratified sample of the training split (a minute or two). We keep the top 25: fewer features make the live sensor and its parity test on day 5 much easier.

In [ ]:
importance = rank_features(train, candidates, seed=SEED)
features = list(importance.index[:TOP_K])

fig, ax = plt.subplots(figsize=(8, 9))
top = importance.head(35)
colors = ["steelblue" if f in features else "lightgray" for f in top.index]
ax.barh(top.index[::-1], top.values[::-1], color=colors[::-1])
ax.set(xlabel="Random Forest importance", title=f"Top 35 features (blue = kept, top {TOP_K})")
plt.tight_layout()
plt.savefig(FIGURES / "02_feature_importance.png", dpi=120)
print(f"top {TOP_K} carry {importance.head(TOP_K).sum():.1%} of total importance")

In [ ]:
corr = train[features].astype(float).corr()
plt.figure(figsize=(11, 9))
sns.heatmap(corr, cmap="vlag", center=0, square=True, cbar_kws={"shrink": 0.7})
plt.title("Correlation of the selected features (train)")
plt.tight_layout()
plt.savefig(FIGURES / "02_selected_corr.png", dpi=120)

## 6. Save

In [ ]:
cols = features + [LABEL_COL, FAMILY_COL]
for name, part in splits.items():
    part[cols].reset_index(drop=True).to_parquet(PROCESSED / f"{name}.parquet", index=False)
save_feature_config(features, importance, dropped_corr, ROOT / "configs/features.yaml")
importance.rename("importance").to_csv(ROOT / "reports/feature_importance.csv")
(ROOT / "reports/split_counts.json").write_text(json.dumps(
    {k: v[FAMILY_COL].value_counts().to_dict() for k, v in splits.items()}, indent=2))
print("saved train/val/test parquet and configs/features.yaml")
features

## Notes for day 3

- Target is `Family` (7 classes). Report macro-F1 and per-class false-positive rate, not accuracy.
- Classes are imbalanced (Bot and Web attack are small): use class weights; try SMOTE on train only if time allows.
- Train on `train.parquet`, tune on `val.parquet`, touch `test.parquet` once at the end.